# Lab 9 — Governance & security in Unity Catalog (Exam section 7 · 15%)

**How this lab works**
- Run cells top to bottom (`Shift+Enter`). After a kernel restart, run again from the top.
- Cells marked `# TODO` contain `<FILL-IN>`: replace it with your code, then run the `check(...)` cell.
- Stuck? Ask Claude in the terminal: *"hint for 1c"*. Solutions are in `notebooks/solutions/` (try first!).
- 🖥 **Local** means open-source Spark + Delta on your Mac. ☁ **Databricks** shows the real syntax for the exam.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # lets the notebook import labutils
from labutils import get_spark, path, reset_lab, show, checkpoint, table_path, landing
from labutils.check import check
from pyspark.sql import functions as F
spark = get_spark()
reset_lab("lab9")
from labutils.submit import submit

## 9a · GRANT and REVOKE privileges

In [ ]:
sql = """
-- 1) group `analysts` must be able to READ main.sales.orders  (3 statements)
GRANT USE CATALOG ON CATALOG main TO `analysts`;
GRANT USE SCHEMA ON SCHEMA main.sales TO `analysts`;
GRANT SELECT ON TABLE main.sales.orders TO `analysts`;
-- 2) group `contractors` must NOT read main.sales.orders anymore
REVOKE SELECT ON TABLE main.sales.orders FROM `contractors`;
-- 3) service principal `etl-sp` must be able to insert/update/delete rows in main.sales.orders
GRANT MODIFY ON TABLE main.sales.orders TO `etl-sp`;
"""
submit("9a", sql)

In [ ]:
check("9a")

## 9b · Managed vs external tables and converting between them

In [ ]:
sql = """
-- 1) create an EXTERNAL table main.raw.events_ext stored at 's3://acme-raw/events'
CREATE TABLE main.raw.events_ext (id INT, ts TIMESTAMP) LOCATION 's3://acme-raw/events';
-- 2) convert it to a MANAGED table
ALTER TABLE main.raw.events_ext SET MANAGED;
-- 3) oops — roll the conversion back (allowed within 14 days)
ALTER TABLE main.raw.events_ext UNSET MANAGED;
"""
answers = {
  "drop_managed_deletes_data": True,
  "drop_external_deletes_data": False,
}
submit("9b", {"sql": sql, **answers})

In [ ]:
check("9b")

## 9c · Column masks and row filters

In [ ]:
from labutils.governance import login
login("you@corp.com", [])     # registers is_account_group_member() for this lab
spark.sql("""CREATE TABLE lab9.customers AS SELECT * FROM VALUES
  (1, 'Ana',  '4111111111111111', 'EU'),
  (2, 'Ben',  '5500000000000004', 'US'),
  (3, 'Chen', '340000000000009',  'EU')
  AS t(id, name, credit_card_number, region)""")

In [ ]:
MASK = """CASE WHEN is_account_group_member('finance_team') THEN credit_card_number
               ELSE concat('****', right(credit_card_number, 4)) END"""
ROW_FILTER = """is_account_group_member('admins') OR region = 'EU'"""

spark.sql(f"""CREATE OR REPLACE TEMP VIEW customers_secure AS
SELECT id, name, {MASK} AS credit_card_number, region FROM lab9.customers WHERE {ROW_FILTER}""")
submit("9c", {"mask": MASK, "row_filter": ROW_FILTER})
for user, groups in [("fiona@corp.com", ["finance_team"]), ("adam@corp.com", ["admins"]), ("eve@corp.com", [])]:
    login(user, groups)
    show(spark.table("customers_secure"))

In [ ]:
check("9c")

## 9d · ABAC policies (attribute-based access control)

In [ ]:
SITUATIONS = {
  "Q1": "500 tables in a catalog contain columns tagged 'pii'. Every new table with a 'pii' column must be masked automatically.",
  "Q2": "One single table needs one column masked for one group.",
  "Q3": "Rows of every table tagged 'region' must be filtered for non-admins, across many schemas.",
  "Q4": "Give analysts permission to read one table.",
}
TOOLS = {
  "ABAC": "An ABAC policy (on catalog/schema) that uses governed tags to apply a mask or row filter to all matching tables",
  "MASK": "A column mask function set on that one table's column",
  "GRANT": "A GRANT SELECT statement",
}
for k, v in SITUATIONS.items(): print(k, "→", v)
print()
for k, v in TOOLS.items(): print(k, "=", v)

In [ ]:
submit("9d", {"Q1": "ABAC", "Q2": "MASK", "Q3": "ABAC", "Q4": "GRANT"})

In [ ]:
check("9d")